# ASGC on Fashion-MNIST - frozen results

**This notebook is a results viewer, not a training run.** It renders every Fashion-MNIST table, figure, and finding without retraining, because the experiments were already executed (single seed, seed 0). The numbers are copied from [`findings_for_report.md`](../findings_for_report.md) sections 3 and 7.

**Provenance.** Research simulation `src/asgc/`; small CNN (55,338 parameters, 221,352 B per uncompressed gradient); 3 worker processes with a `spawn` start method; worker 1 delayed by +0.3 s; async budget 3000 updates, sync budget 1500 rounds (4500 worker gradients); SGD with server-side step decay; RTX 4050 laptop GPU, one torch thread per process. Runs were written to `results/fmnist_*` (gitignored) with `events.jsonl` as the source of truth.

**What "frozen" means.** Every cell below reads embedded constants, so it runs on a laptop or Colab with no GPU, no dataset download, and no experiment. Reproduction commands are at the end if you ever want to regenerate the raw logs.


## 1. Headline table - unconstrained network

Async methods use 3000 updates; synchronous SGD uses 1500 rounds. `time_to_0.85` is the wall-clock to 85% test accuracy. Compression ratio (CR) is exact encoded bytes relative to the uncompressed up-link; the harness does not model a down-link.


In [ ]:
import pandas as pd

# Frozen from findings_for_report.md section 3 (single seed, seed 0).
headline = [
    {"method": "Synchronous SGD",               "accuracy": 86.11, "loss": 0.393, "gb": 0.996, "cr": 1.00, "upd_s": 2.7,  "time_to_085_s": 274.3},
    {"method": "Plain async",                   "accuracy": 87.20, "loss": 0.352, "gb": 0.664, "cr": 1.00, "upd_s": 44.7, "time_to_085_s": 27.8},
    {"method": "Async + fixed staleness",       "accuracy": 87.03, "loss": 0.360, "gb": 0.706, "cr": 1.00, "upd_s": 42.2, "time_to_085_s": 41.1},
    {"method": "Async + fixed Top-K",           "accuracy": 86.92, "loss": 0.369, "gb": 0.332, "cr": 2.00, "upd_s": 40.4, "time_to_085_s": 31.5},
    {"method": "Async + Top-K + EF",            "accuracy": 87.47, "loss": 0.349, "gb": 0.332, "cr": 2.00, "upd_s": 38.5, "time_to_085_s": 32.4},
    {"method": "Adaptive (pre-fix controller)", "accuracy": 87.05, "loss": 0.358, "gb": 0.470, "cr": 1.52, "upd_s": 36.2, "time_to_085_s": 40.8},
]
headline_df = pd.DataFrame(headline)
headline_df["speedup_vs_sync"] = headline_df.loc[0, "time_to_085_s"] / headline_df["time_to_085_s"]
headline_df.style.format({
    "accuracy": "{:.2f}%", "loss": "{:.3f}", "gb": "{:.3f}",
    "cr": "{:.2f}x", "upd_s": "{:.1f}", "time_to_085_s": "{:.1f} s",
    "speedup_vs_sync": "{:.1f}x",
}).highlight_max(subset=["accuracy"], color="#d7f5d7").highlight_min(subset=["gb"], color="#d7f5d7")


### Findings from this table

- **F1 - Asynchrony is worth 9.9x wall-clock (16.3x throughput) under stragglers.** Plain async reached 85% accuracy in 27.8 s versus 274.3 s for averaged synchronous SGD. Fairness caveat: sync performed 4500 worker gradient computations to async's 3000 and still took 8x longer in wall-clock.
- **F2 - Top-K + EF is the Pareto winner at every operating point tested.** Best accuracy of all six methods (87.47%) at exactly half the up-link bytes; it meets the 50% communication gate exactly.
- **F3 - Staleness control is insurance, not speed.** Fixed staleness sends *more* bytes than plain async (0.706 vs 0.664 GB) because rejected gradients were still transmitted, and converges slightly slower. Its value is bounding applied staleness to at most 4 while plain async applied tau up to 19.
- **F4 - Fixed compression without error feedback costs accuracy:** 86.92% vs 87.47% at the same bytes.


## 2. Controller A/B/C - why the adaptive arm was fixed

The first adaptive arm underperformed because its instability signal was raw loss spread: every "instability" trip happened while loss was *falling*, so the controller punished fast learning and ratcheted compression up to only 1.52x. Replacing raw spread with the **mean positive loss rebound** (average upward move between consecutive evaluations) fixed it. The three rows below use the same config; only the signal differs.


In [ ]:
# Frozen from findings_for_report.md section 7 (FMNIST, single seed).
controller = [
    {"controller": "A: raw spread",            "accuracy": 87.05, "gb": 0.470, "cr": 1.52, "time_to_085_s": 40.8},
    {"controller": "B: rebound (canonical)",   "accuracy": 87.09, "gb": 0.195, "cr": 3.65, "time_to_085_s": 36.2},
    {"controller": "C: rebound + persistence", "accuracy": 86.92, "gb": 0.196, "cr": 3.64, "time_to_085_s": 42.8},
]
controller_df = pd.DataFrame(controller)
controller_df.style.format({
    "accuracy": "{:.2f}%", "gb": "{:.3f}", "cr": "{:.2f}x", "time_to_085_s": "{:.1f} s",
}).highlight_min(subset=["gb", "time_to_085_s"], color="#d7f5d7")


- **F11 - With the signal fixed, the adaptive method beats the best fixed policy on bytes:** 0.195 vs 0.332 GB (-41%) at equal accuracy (within noise). This is the first FMNIST configuration where the adaptive machinery earns its overhead.
- **F12 - Persistence never engaged after the fix** (no false unstable windows to gate); it is retained for CIFAR-class workloads where genuine instability is expected.
- Method note: this diagnosis was possible only because controller decisions are logged with their input statistics - the event-sourced audit trail paid for itself.


## 3. Figures from the frozen table

Two views of the same data: final accuracy (higher is better) and up-link bytes (lower is better). The plot is generated from the embedded constants, not from a rerun.


In [ ]:
import matplotlib.pyplot as plt

plot_df = headline_df.copy()
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
colors = ["#9aa5b1", "#4c78a8", "#4c78a8", "#4c78a8", "#2e7d32", "#f58518"]
short = ["sync", "plain\nasync", "fixed\nstale", "fixed\nTop-K", "Top-K\n+ EF", "adaptive\n(pre-fix)"]

axes[0].bar(short, plot_df["accuracy"], color=colors)
axes[0].set_ylim(84, 88.2)
axes[0].set_ylabel("test accuracy (%)")
axes[0].set_title("Fashion-MNIST final accuracy")
for i, v in enumerate(plot_df["accuracy"]):
    axes[0].text(i, v + 0.06, f"{v:.2f}", ha="center", fontsize=8)

axes[1].bar(short, plot_df["gb"], color=colors)
axes[1].set_ylabel("up-link GB")
axes[1].set_title("Fashion-MNIST bytes sent")
for i, v in enumerate(plot_df["gb"]):
    axes[1].text(i, v + 0.015, f"{v:.3f}", ha="center", fontsize=8)

for ax in axes:
    ax.spines[["top", "right"]].set_visible(False)
    ax.tick_params(axis="x", labelsize=8)
fig.tight_layout()
plt.show()


## 4. Caveats specific to these runs

- **Single seed.** Headline differences are within roughly a 0.5-point noise band; treat orderings among the async methods as indicative.
- **Sync semantics.** The synchronous row is the research harness's `workload.sync` mode: pushes are buffered per round, the round's gradients are **averaged**, and one server step is applied. This is not the same as the transformer notebook's barrier-serialized per-push sync (findings section 12, item 3).
- **Rollback scope.** The harness has fire-and-forget pushes with no acknowledgement, so a rejected stale update cannot roll its residual back. Rejection still blocks application and advances no version. Residual rollback is implemented and unit-tested in `src/torch_asgc`, not in these runs (findings section 12, item 4).
- **Byte model.** Only the compressed up-link is charged; down-link snapshots are unmodeled and identical across methods, so "GB" here means up-link GB. The controller signal in the harness is evaluation loss, not per-push training loss.
- **`time_to_0.85`** is available for every row because the workload crosses 85%; CIFAR's target is 0.80 (see `cifar_asgc.ipynb`).


## 5. Reproduce from raw telemetry (optional)

These commands rerun the experiments and are **not** executed by this notebook. They require the repository and `pip install -r requirements.txt`.

```bash
# Individual headline methods
python scripts/run_experiment.py --config configs/comparisons/fmnist_headline/sync_sgd.yaml
python scripts/run_experiment.py --config configs/comparisons/fmnist_headline/async_plain.yaml
python scripts/run_experiment.py --config configs/comparisons/fmnist_headline/async_fixed_staleness.yaml
python scripts/run_experiment.py --config configs/comparisons/fmnist_headline/async_fixed_topk.yaml
python scripts/run_experiment.py --config configs/comparisons/fmnist_headline/async_topk_ef.yaml
python scripts/run_experiment.py --config configs/comparisons/fmnist_headline/adaptive.yaml

# All six methods via the sequential matrix driver
python scripts/run_matrix.py --matrix configs/matrix_fmnist_headline.yaml

# Aggregate runs into a summary table and figures
python scripts/plot_results.py --runs fmnist_sync_sgd fmnist_async_plain fmnist_async_fixed_staleness fmnist_async_fixed_topk fmnist_async_topk_ef fmnist_adaptive --target 0.85
```

`run_experiment.py` writes `results/<run.name>/events.jsonl`; `plot_results.py` reads runs by name and writes `results/plot/summary.csv` plus comparison figures.
